
# EE 467 Final Project  
## Botnet Detection Using Machine Learning  

**Team Members:** Ethan Le, Dhruv Naik  

This notebook implements:
- Binary botnet detection (Botnet vs Non-Botnet)
- Scenario-based train/test splitting
- Leakage control
- Logistic Regression baseline
- Random Forest + XGBoost
- Autoencoder anomaly baseline
- Recall @ Fixed FPR evaluation



## 1. Imports and Configuration

This section loads required libraries and sets random seeds for reproducibility.
Libraries follow the same structure and style used in prior EE 467 labs.


In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight

import xgboost as xgb

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)



## 2. Data Loading

We load CTU-13 NetFlow CSV files for selected scenarios.
We train on one scenario and test on an unseen scenario to evaluate generalization.


In [ ]:

DATA_DIR = "path_to_ctu13_csv_files"

def load_scenario(path):
    return pd.read_csv(path)

# Example usage:
# scenario_train = load_scenario("scenario_1.csv")
# scenario_test = load_scenario("scenario_2.csv")



## 3. Label Processing (Binary Setup)

We merge:
- Botnet + C&C → 1
- Normal → 0

Background handling can be modified depending on experiment design.


In [ ]:

def process_labels(df):
    df = df.copy()
    df['binary_label'] = df['label'].apply(
        lambda x: 1 if x in ['Botnet', 'C&C'] else 0
    )
    return df



## 4. Feature Engineering and Leakage Control

We:
- Add rate-based features
- Remove raw IP addresses to prevent shortcut learning
- Avoid data leakage between scenarios


In [ ]:

DROP_COLUMNS = ['label', 'src_ip', 'dst_ip']

def feature_engineering(df):
    df = df.copy()
    df['bytes_per_sec'] = df['bytes'] / (df['duration'] + 1e-6)
    df['pkts_per_sec'] = df['packets'] / (df['duration'] + 1e-6)
    return df

def prepare_features(df):
    df = df.drop(columns=DROP_COLUMNS, errors='ignore')
    X = df.drop(columns=['binary_label'])
    y = df['binary_label']
    return X, y



## 5. Scenario-Based Train/Validation/Test Split

We:
- Train on Scenario A
- Validate using 20% of training data
- Test on unseen Scenario B

This evaluates cross-scenario generalization.


In [ ]:

# Example split structure:

# train_df = process_labels(feature_engineering(scenario_train))
# test_df  = process_labels(feature_engineering(scenario_test))

# X_train, y_train = prepare_features(train_df)
# X_test, y_test   = prepare_features(test_df)

# X_train, X_val, y_train, y_val = train_test_split(
#     X_train, y_train,
#     test_size=0.2,
#     stratify=y_train,
#     random_state=RANDOM_STATE
# )



## 6. Scaling and Class Imbalance

We:
- Fit scaler ONLY on training data
- Compute class weights ONLY on training fold


In [ ]:

# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_val_scaled   = scaler.transform(X_val)
# X_test_scaled  = scaler.transform(X_test)

# classes = np.unique(y_train)
# weights = compute_class_weight(
#     class_weight='balanced',
#     classes=classes,
#     y=y_train
# )
# class_weight_dict = dict(zip(classes, weights))



## 7. Evaluation Utilities

We prioritize low False Positive Rate (FPR).

Primary metric:
- Recall @ Fixed FPR (e.g., 1%)

Thresholds are selected using validation data only.


In [ ]:

def recall_at_fixed_fpr(y_true, y_scores, target_fpr=0.01):
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    idx = np.where(fpr <= target_fpr)[0]
    if len(idx) == 0:
        return 0.0
    return tpr[idx[-1]]

def select_threshold_at_fpr(y_true, y_scores, target_fpr=0.01):
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    idx = np.where(fpr <= target_fpr)[0]
    if len(idx) == 0:
        return 0.5
    return thresholds[idx[-1]]



## 8. Logistic Regression Baseline

This serves as:
- Interpretable baseline
- Linear probabilistic classifier


In [ ]:

# log_model = LogisticRegression(
#     class_weight=class_weight_dict,
#     max_iter=1000,
#     random_state=RANDOM_STATE
# )

# log_model.fit(X_train_scaled, y_train)
# val_probs = log_model.predict_proba(X_val_scaled)[:, 1]
# threshold = select_threshold_at_fpr(y_val, val_probs)

# test_probs = log_model.predict_proba(X_test_scaled)[:, 1]

# print("ROC-AUC:", roc_auc_score(y_test, test_probs))
# print("Recall@1%FPR:", recall_at_fixed_fpr(y_test, test_probs))



## 9. Tree-Based Models

We implement:
- Random Forest
- XGBoost (boosted trees)

These capture nonlinear feature interactions.


In [ ]:

# rf_model = RandomForestClassifier(
#     n_estimators=200,
#     class_weight=class_weight_dict,
#     random_state=RANDOM_STATE
# )

# rf_model.fit(X_train, y_train)

# xgb_model = xgb.XGBClassifier(
#     n_estimators=300,
#     max_depth=6,
#     learning_rate=0.05,
#     random_state=RANDOM_STATE
# )

# xgb_model.fit(X_train, y_train)



## 10. Autoencoder (Anomaly Detection Baseline)

We:
- Train ONLY on normal traffic
- Use reconstruction error as anomaly score
- Select threshold at fixed FPR


In [ ]:

class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 16)
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)
